In [1]:
import pandas as pd
import numpy as np
import gzip
import matplotlib.pyplot as plt
import pickle
from scipy.stats import linregress
import os
import sys
from datetime import date
from sklearn.metrics import auc, f1_score
import random
from sklearn.decomposition import NMF, non_negative_factorization
import seaborn as sns
import time
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1 import axes_grid

In [4]:
tsv_file = open("/home/ctrader/Hold/DHS/DHS_Index_and_Vocabulary_metadata.tsv")
df = pd.read_csv(tsv_file, delimiter="\t")

In [7]:
biosample_id_order = np.array(df['Altius Biosample ID'])
extra_tsv_file = open('/home/meuleman/work/projects/ENCODE3/WM20180608_733_sample_metadata/ENCODE 733-sample list WM20180608 - ENCODE3 metadata.tsv')
extra_biosample_df = pd.read_csv(extra_tsv_file, delimiter="\t")
extra_biosample_df = extra_biosample_df[~extra_biosample_df['BAM file'].isnull()]
# extra_biosample_df['ind'] = extra_biosample_df.index
extra_biosample_df = extra_biosample_df.set_index(['DS_plus'])
extra_biosample_df = extra_biosample_df.loc[biosample_id_order]
extra_biosample_df['ind'] = np.arange(733)
extra_biosample_df['DS_plus'] = extra_biosample_df.index
extra_biosample_df = extra_biosample_df.set_index(['ind'])

In [18]:
pd.set_option('display.max_columns', None)
extra_biosample_df[extra_biosample_df['DCC_experiment']=='ENCSR000EMZ']

,Included in 827-sample set?,Included in 665-sample set?,DCC SPOT1 >= 0.3?,DCC SPOT1 >= 0.25,DCC SPOT1 >= 0.2,TO BE INCLUDED,Candidate removal -- insert sizes & dupe rate,Candidate removal -- insert sizes,AG,LN,DS,AG_plus,LN_plus,DCC_experiment,DCC_library,DCC_biosample,repls,taxonomy,display name,class,system,subsystem,organ,germ layer,Description,Growth stage,Biological_state,Sample_group,Age,Sex,Ethnicity,Donor_ID,Alt_donor_ID,Unique cellular condition,Grant,Status,DCC SPOT1,SPOT1,SPOT2,SPOT2_num_spots,SPOT2_num_bases,preseq_max,nuclear_dupes,reads_total,reads_adaptercontamination,reads_lowquality,reads_nonnuclear,duplication rate,insert_spike,insert_ratio,lib_submission_date,lib_kit_method,lib_cleanup,lib_dhs_units,lib_frag_analyzer_conc,lib_nuclei_acid_ng,sample_creation_date,sample_dhs_time_minutes,sample_dhs_mg_per_mil_nuclei,sample_dhs_volume_ml,sample_dhs_percent_igepal,sample_dhs_percent_loss,sample_dhs_nuclei_count,sample_dhs_dnasei_units,sample_dhs_protease_inhibitor,tissueculture_creation_date,fractionation_id,fractionation_method,fractionation_yield,fractionation_num_nuclei,BAM file,DS_plus
ind,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
44,Yes,Yes,Yes,Yes,Yes,Yes,No,No,4188.0,2365.0,11909.0,AG4188,LN2365,ENCSR000EMZ,ENCLB449ZZZ,ENCBS293AAA,DS13133,H7-hESC,hESCT0,Lines,Embryonic,NaN,Germ,Primitive,H7 embryonic stem,Embryonic,ESC,Lines,NaN,F,NaN,NaN,NaN,1,ENCODE2,published,0.6453,0.5892,0.6183,432610.0,145512356.0,351182398.0,100260481.0,401363495.0,831.0,102696753.0,57874824.0,0.41638,NaN,NaN,2009-08-19,PE,minElute,80.0,NaN,14,2010-08-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/net/seq/data/aggregations/LN2365/aggregation-...,DS11909
110,Yes,Yes,Yes,Yes,Yes,Yes,No,No,4132.0,2721.0,13133.0,AG4132,LN2721,ENCSR000EMZ,ENCLB450ZZZ,ENCBS297CQV,DS11909,HESC,hESCT0,Lines,Embryonic,NaN,Germ,Primitive,H7 embryonic stem,Embryonic,ESC,Lines,NaN,F,NaN,NaN,NaN,0,ENCODE2,published,0.5993,0.5538,0.3722,100250.0,27314733.0,332380820.0,3475713.0,32401603.0,13.0,7883024.0,2489666.0,0.15778,NaN,NaN,2009-11-16,PE,minElute,120.0,NaN,31.2,2010-08-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/net/seq/data/aggregations/LN2721/aggregation-...,DS13133


In [17]:
extra_biosample_df

,Included in 827-sample set?,Included in 665-sample set?,DCC SPOT1 >= 0.3?,DCC SPOT1 >= 0.25,DCC SPOT1 >= 0.2,TO BE INCLUDED,Candidate removal -- insert sizes & dupe rate,Candidate removal -- insert sizes,AG,LN,DS,AG_plus,LN_plus,DCC_experiment,DCC_library,DCC_biosample,repls,taxonomy,display name,class,system,subsystem,organ,germ layer,Description,Growth stage,Biological_state,Sample_group,Age,Sex,Ethnicity,Donor_ID,Alt_donor_ID,Unique cellular condition,Grant,Status,DCC SPOT1,SPOT1,SPOT2,SPOT2_num_spots,SPOT2_num_bases,preseq_max,nuclear_dupes,reads_total,reads_adaptercontamination,reads_lowquality,reads_nonnuclear,duplication rate,insert_spike,insert_ratio,lib_submission_date,lib_kit_method,lib_cleanup,lib_dhs_units,lib_frag_analyzer_conc,lib_nuclei_acid_ng,sample_creation_date,sample_dhs_time_minutes,sample_dhs_mg_per_mil_nuclei,sample_dhs_volume_ml,sample_dhs_percent_igepal,sample_dhs_percent_loss,sample_dhs_nuclei_count,sample_dhs_dnasei_units,sample_dhs_protease_inhibitor,tissueculture_creation_date,fractionation_id,fractionation_method,fractionation_yield,fractionation_num_nuclei,BAM file,DS_plus
ind,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No,5636.0,1203.0,7748.0,AG5636,LN1203,ENCSR000EMQ,ENCLB435ZZZ,ENCBS057ENC,DS7784,GM06990,GM06990,Lines,Hematopoietic,Lymphoid,Blood,Mesoderm,Lymphoblastoid,Adult,Immortalized,Lines,41Y,F,NaN,NaN,NaN,0,ENCODE3,unpublished,0.6790,0.6545,0.4433,111013.0,49120551.0,320807053.0,76385327.0,238189351.0,0.0,81698194.0,13809567.0,0.535355,NaN,NaN,2009-02-23,NaN,Sucrose,NaN,NaN,50,2010-08-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/net/seq/data/aggregations/LN1203/aggregation-...,DS7748
1,Yes,Yes,Yes,Yes,Yes,Yes,No,No,5635.0,1207.0,7764.0,AG5635,LN1207,ENCSR000ENP,ENCLB480ZZZ,ENCBS114ENC,DS7768,h.HepG2,HepG2,Lines,Hepatic,NaN,Liver,Endoderm,hepatocellular carcinoma,Child,Cancer,Cancer,15Y,M,NaN,NaN,NaN,1,ENCODE2,published,0.5858,0.5618,0.4482,115158.0,47318563.0,386655881.0,61507725.0,248320583.0,39.0,81883686.0,27610555.0,0.443055,NaN,NaN,2009-02-23,NaN,Sucrose,NaN,NaN,50,2010-08-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/net/seq/data/aggregations/LN1207/aggregation-...,DS7764
2,Yes,Yes,Yes,Yes,Yes,Yes,No,No,5634.0,1222.0,7840.0,AG5634,LN1222,ENCSR000EQC,ENCLB591ZZZ,ENCBS345AAA,NaN,hTH1,hTH1,Primary,Hematopoietic,T-cell,Blood,Mesoderm,T1 helper,Adult,Primary,Primary,25Y?,F?,NaN,Wilson01,NaN,1,ENCODE2,published,0.6470,0.6243,0.5285,111660.0,54584608.0,365529941.0,70173889.0,232708777.0,0.0,63279764.0,20270380.0,0.470465,NaN,NaN,2007-06-06,NaN,Sucrose,6.0,NaN,534.9,2010-08-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/net/seq/data/aggregations/LN1222/aggregation-...,DS7840
3,Yes,Yes,Yes,Yes,Yes,Yes,No,No,4219.0,1264.0,8200.0,AG4219,LN1264,ENCSR000ENO,ENCLB479ZZZ,ENCBS890POO,NaN,Hela,Hela,Lines,Genitourinary,NaN,Uterus,Ectoderm,Cervical carcinoma,Adult,Cancer,Cancer,31Y,F,NaN,NaN,NaN,1,ENCODE2,published,0.6444,0.6217,0.5260,81123.0,26493440.0,268974910.0,6376801.0,44192770.0,19.0,10604424.0,10215622.0,0.272831,NaN,NaN,2007-08-24,NaN,new Sucrose,4.0,NaN,50,2010-08-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/net/seq/data/aggregations/LN1264/aggregation-...,DS8200
4,Yes,Yes,Yes,Yes,Yes,Yes,No,No,4218.0,1269.0,8235.0,AG4218,LN1269,ENCSR000EMI,ENCLB422ZZZ,ENCBS391ENC,DS8416,CACO2,CACO2,Lines,Digestive,NaN,Colon,Endoderm,Colorectal adenocarcinoma,Adult,Cancer,Cancer,72Y,M,NaN,NaN,NaN,1,ENCODE2,published,0.7190,0.6829,0.5102,36310.0,14966147.0,106079631.0,9441776.0,67349236.0,104.0,43269847.0,1319330.0,0.414842,NaN,NaN,2007-09-05,NaN,Sucrose,8.0,NaN,1,2012-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/net/seq/data/aggregations/LN1269/aggregation-...,DS8235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..